### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Load all excel files

In [2]:
import pandas as pd
import os

data_path = "../data/flipkart data"

files = [
    "flipkart_laptops.csv",
    "flipkart_mobiles.csv",
    "flipkart_refrigerator.csv",
    "flipkart_smart_watch.csv",
    "flipkart_tv.csv",
    "flipkart_washing_machine.csv"
]

dfs = []

for file in files:
    temp_df = pd.read_csv(
        os.path.join(data_path, file),
        encoding='latin1'
    )
    dfs.append(temp_df)

fp_data = pd.concat(dfs, ignore_index=True)

print(fp_data.head())

                                                Name   Brand  Selling Price  \
0  Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 ...  Lenovo          36990   
1  Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/512 ...  Lenovo          37990   
2  ASUS VivoBook 15 (2022) Core i3 10th Gen - (8 ...    ASUS          32990   
3  realme Book (Slim) Core i3 11th Gen - (8 GB/25...  realme          46990   
4  DELL Inspiron Core i3 11th Gen - (8 GB/1 TB HD...    DELL          38990   

     MRP Discount  Ratings                 No_of_ratings  \
0  60890  39% off      4.2      670 Ratings & 54 Reviews   
1  59390  36% off      4.2    3803 Ratings & 362 Reviews   
2  45990  28% off      4.3    8727 Ratings & 876 Reviews   
3  54999  14% off      4.4  11894 Ratings & 1773 Reviews   
4  61202  36% off      4.3        65 Ratings & 6 Reviews   

                                             Details  
0  ['Intel Core i3 Processor (11th Gen)' '8 GB DD...  
1  ['Intel Core i3 Processor (11th Gen)' '8 GB DD...  

### Dataset Overview

In [3]:
fp_data.head()

,Name,Brand,Selling Price,MRP,Discount,Ratings,No_of_ratings,Details
0,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 ...,Lenovo,36990,60890,39% off,4.2,670 Ratings & 54 Reviews,['Intel Core i3 Processor (11th Gen)' '8 GB DD...
1,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/512 ...,Lenovo,37990,59390,36% off,4.2,3803 Ratings & 362 Reviews,['Intel Core i3 Processor (11th Gen)' '8 GB DD...
2,ASUS VivoBook 15 (2022) Core i3 10th Gen - (8 ...,ASUS,32990,45990,28% off,4.3,8727 Ratings & 876 Reviews,['Intel Core i3 Processor (10th Gen)' '8 GB DD...
3,realme Book (Slim) Core i3 11th Gen - (8 GB/25...,realme,46990,54999,14% off,4.4,11894 Ratings & 1773 Reviews,['Stylish & Portable Thin and Light Laptop' '1...
4,DELL Inspiron Core i3 11th Gen - (8 GB/1 TB HD...,DELL,38990,61202,36% off,4.3,65 Ratings & 6 Reviews,['Processor: Intel i3-1115G4 (Base- 1.70 GHz &...


In [4]:
fp_data.shape

(2525, 8)

In [5]:
fp_data.columns

Index(['Name', 'Brand', 'Selling Price', 'MRP', 'Discount', 'Ratings',
       'No_of_ratings', 'Details'],
      dtype='object')

In [6]:
fp_data.isnull().sum()

Name             0
Brand            0
Selling Price    0
MRP              0
Discount         0
Ratings          0
No_of_ratings    0
Details          0
dtype: int64

### From Overview

1. Shape of the dataset is (2525, 8)
2. column names ['Name', 'Brand', 'Selling Price', 'MRP', 'Discount', 'Ratings',
       'No_of_ratings', 'Details']
3. there is no missing values in this dataset

## Phase 2

### Data Processing 

**Now we prepare text features for recommendation modeling.**

####  Goal

We'll create a combined product profile using:
* Name
* Brand
* Details

1. Then convert text into vectors using **TF-IDF**
2. Then use cosine similarity to recommend products.
3. This is how many recommendation systems work.

### Create combined features

In [7]:
fp_data["combined_features"] = (
    fp_data["Name"].astype(str) + " " +
    fp_data["Brand"].astype(str) + " " +
    fp_data["Details"].astype(str)
)

In [8]:
# Verify

fp_data[["Name","combined_features"]].head()

,Name,combined_features
0,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 ...,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 ...
1,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/512 ...,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/512 ...
2,ASUS VivoBook 15 (2022) Core i3 10th Gen - (8 ...,ASUS VivoBook 15 (2022) Core i3 10th Gen - (8 ...
3,realme Book (Slim) Core i3 11th Gen - (8 GB/25...,realme Book (Slim) Core i3 11th Gen - (8 GB/25...
4,DELL Inspiron Core i3 11th Gen - (8 GB/1 TB HD...,DELL Inspiron Core i3 11th Gen - (8 GB/1 TB HD...


### Remove Duplicate products

In [9]:
fp_data.drop_duplicates(subset="Name", inplace=True)

In [10]:
fp_data.shape

(2205, 9)

## Phase 3

### TF-IDF Features engineering

**Now we Convert text into machine-readable vectors.**

### TF-IDF Vectroization

In [11]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(
    fp_data["combined_features"]
)

In [12]:
# Matrix Shape

tfidf_matrix.shape

(2205, 3350)

### Phase 4 - Similarity Modeling

**This is the core recommendation engine**

### Cosine Similarity

In [13]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [14]:
# Similarity matxic shape

cosine_sim.shape


(2205, 2205)

In [15]:
type(cosine_sim)

numpy.ndarray

## Phase 5 - Build Recommendation Function

**Now we create the actual AI Recommendation engine**

### Recommendation Function

In [16]:
indices = pd.Series(
    fp_data.index,
    index=fp_data["Name"]
).drop_duplicates()


def recommend_products(product_name, top_n=5):

    if product_name not in indices:
        return "Product not found"

    idx = indices[product_name]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    product_indices = [
        i[0] for i in similarity_scores
    ]

    recommendations = fp_data.iloc[product_indices][[
        "Name",
        "Brand",
        "Selling Price",
        "Ratings"
    ]]

    recommendations["Similarity Score"] = [
        round(score[1] * 100, 2)
        for score in similarity_scores
    ]

    return recommendations

### Test Recommendation Engine

In [17]:
fp_data["Name"].iloc[0]

'Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 GB SSD/Windows 11 Home) 14ITL05 Thin and Light Laptop'

In [18]:
recommend_products("Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 GB SSD/Windows 11 Home) 14ITL05 Thin and Light Laptop")

,Name,Brand,Selling Price,Ratings,Similarity Score
175,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 ...,Lenovo,35990,4.2,84.51
50,Lenovo IdeaPad 3 Core i3 10th Gen - (8 GB/256 ...,Lenovo,33990,4.3,75.24
1,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/512 ...,Lenovo,37990,4.2,75.12
342,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 ...,Lenovo,36990,4.3,73.82
108,Lenovo IdeaPad 3 Core i3 10th Gen - (8 GB/256 ...,Lenovo,33990,4.3,71.06


## Phase 6 Explainable Recommendation Intelligence

* now i make this project look professional and bussiness - oriented.
* Instead of showing only products, i explain WHY they are recommended.
* This massively imporves portfolio quality

### Enhanced Recommendation Function

In [19]:
def recommend_products(product_name, top_n=5):

    if product_name not in indices:
        return "Product not found"

    idx = indices[product_name]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    product_indices = [
        i[0] for i in similarity_scores
    ]

    recommendations = fp_data.iloc[product_indices][[
        "Name",
        "Brand",
        "Selling Price",
        "Ratings"
    ]].copy()

    recommendations["Similarity Score"] = [
        round(score[1] * 100, 2)
        for score in similarity_scores
    ]

    explanations = []

    for score in recommendations["Similarity Score"]:

        if score > 80:
            explanations.append(
                "Highly similar product with nearly identical specifications"
            )

        elif score > 70:
            explanations.append(
                "Strong recommendation based on product features and category"
            )

        else:
            explanations.append(
                "Moderately related product recommendation"
            )

    recommendations["Recommendation Reason"] = explanations

    return recommendations

In [20]:
recommend_products(
    "Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 GB SSD/Windows 11 Home) 14ITL05 Thin and Light Laptop"
)

,Name,Brand,Selling Price,Ratings,Similarity Score,Recommendation Reason
175,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 ...,Lenovo,35990,4.2,84.51,Highly similar product with nearly identical s...
50,Lenovo IdeaPad 3 Core i3 10th Gen - (8 GB/256 ...,Lenovo,33990,4.3,75.24,Strong recommendation based on product feature...
1,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/512 ...,Lenovo,37990,4.2,75.12,Strong recommendation based on product feature...
342,Lenovo IdeaPad 3 Core i3 11th Gen - (8 GB/256 ...,Lenovo,36990,4.3,73.82,Strong recommendation based on product feature...
108,Lenovo IdeaPad 3 Core i3 10th Gen - (8 GB/256 ...,Lenovo,33990,4.3,71.06,Strong recommendation based on product feature...


## Phase 7 - Export Model Artifacts

**we now save everything for deployment.**

In [21]:
joblib.dump(
    fp_data,"../models/products_data.pkl"
)








['../models/products_data.pkl']

In [26]:
joblib.dump(
    cosine_sim,
    "../models/cosine_similarity.pkl"
)

['../models/cosine_similarity.pkl']

In [23]:
# Indices
joblib.dump(
    fp_data,"../models/indices.pkl"
)

['../models/indices.pkl']

In [27]:
# verify

os. listdir("../models")

['.ipynb_checkpoints',
 'cosine_similarity.pkl',
 'indices.pkl',
 'products_data.pkl']

In [28]:
test = joblib.load(
    "../models/cosine_similarity.pkl"
)

type(test)

numpy.ndarray